Здравствуйте, коллеги!. В этом блокноте я буду работать с заполнением таблиц suppliers,products и orderitem

In [86]:
import glob
import re
import pandas as pd
import random
import numpy as np
from datetime import timedelta,datetime,date
import json
from sqlalchemy import create_engine
import pickle
import openpyxl

In [26]:
df_suppliers=pd.read_csv(r'SQL  database\Service files\suppliers.csv')

In [29]:
# Смотрим данные
df_suppliers

,name,contact_num,email,adress
0,"ООО ""Молочная ферма""",+7 (900) 123-45-67,milkfarm@example.com,"Россия, с. Коровино, ул. Лугавая, 1"
1,"АО ""ИздПресс""",+7 (495) 123-45-00,press@example.com,"Россия, Москва, ул. Печать, 12"
2,PetGoods Ltd.,+44 (20) 1234-5678,info@petgoods.co.uk,"Великобритания, Лондон, 21 Pet Street"
3,Игрушки+,+7 (812) 555-23-45,toysplus@example.com,"Россия, Санкт-Петербург, Невский пр., 77"


In [22]:
# У нас есть уже готовая функция очистки телефона. Модифицируем и проименим
def clean_formated_phone(string):
    if pd.isna(string):
        return None
    else:
        res=re.sub(r'\D','',string)
    if len(res)==11:
        phone=f'+7 ({res[1:4]}) {res[4:7]}-{res[7:9]}-{res[9:11]}'
    elif len(res)==12:
        phone=f'+44 ({res[2:4]}) {res[4:8]}-{res[8:]}'
    else: 
        return None
    return phone

In [28]:
df_suppliers['contact_num']=df_suppliers['contact_num'].apply(clean_formated_phone)

In [31]:
df_suppliers.columns=['name', 'contact_num', 'email', 'address']

In [34]:
with open (r'SQL  database\Service files\configDB.json','r', encoding='UTF-8') as file:
    data=json.load(file)

In [35]:
#Cоздадим параметры подключения
connect = f"postgresql+psycopg2://{data['user']}:{data['password']}@{data['host']}:{data['port']}/{data['db_base']}"

# Создаём движок
engine = create_engine(connect)

In [36]:
#Сохраню я наш получившийся файл
df_suppliers.to_pickle(r'SQL  database\Service files\Генерационные слайды\suppliers.plk')

In [ ]:
df_suppliers.to_sql('suppliers',engine,schema='my_schema',if_exists='append',index=False)


Переходим к таблице Products

In [114]:
df=pd.read_csv(r'SQL  database\Service files\Генерационные слайды\df_start1.csv',parse_dates=['order_date','delivery_date'],dtype={'gift_card':'str'})

In [115]:
df.to_pickle(r'SQL  database\Service files\Генерационные слайды\df_start.pkl')

In [101]:
# Оставим только нужные строки
df=df[[ 'product_id', 'product_name', 'quantity', 'total_amount',
       'discount_amount', ]].sort_values(by='product_id')

In [102]:
# создадим maping для создания кодировки товара в категорию
maping={'Молоко':'Молочная продукция', 'Подписка на газету':'Пресса', 'Корм для собак':'Корма ддля животных', 'Игрушка мяч':'Прочее',
       'Амуниция для собак':'Товары для животных'}
df['category']=df['product_name'].map(maping)

In [103]:

# создадим maping для создания кодировки товара в поставщика
maping1={'Молоко':1, 'Подписка на газету':2, 'Корм для собак':3, 'Игрушка мяч':4,
       'Амуниция для собак':3}
df['supplier_id']=df['product_name'].map(maping1)

In [104]:
# Соберем необходимые поля
df['price']=round(df['total_amount']/df['quantity'],2)
df['allow_backoder']=True
df['discounted']=np.where(df['discount_amount']>0,True,False)
df['discounted_percent']=round(df['discount_amount']/df['total_amount']*100,1)
df['max_order_quality']=4

In [105]:
df.drop_duplicates(subset='product_name',inplace=True,ignore_index=True)

In [106]:
df=df[['product_name', 'category','price',  'supplier_id',  'allow_backoder',
       'discounted', 'discounted_percent', 'max_order_quality']]

In [107]:
df.loc[:, 'expiration_date'] = date.today()

In [108]:
# Перезапишем
df=df[['product_name', 'category', 'price', 'allow_backoder',
       'discounted', 'discounted_percent','expiration_date','supplier_id', 'max_order_quality',]]
     

In [109]:
# Переименуем колонки
df.columns=['name', 'category', 'price', 'allow_backorder', 'discounted',
       'discounted_percent', 'expiration_date', 'supplier_id',
       'max_order_quality']

In [110]:
# Теперь переименуем файли и сохраним в pickle
df_products=df
df_products.to_pickle(r'SQL  database\Service files\Генерационные слайды\df_products.plk')

In [ ]:
# И заливаем
df_products.to_sql('products',engine,schema='my_schema',if_exists='append',index=False)
# Добавил ограничение на уникальность записи отработало



Перехожу к таблице OrderItem

In [159]:
# унас уже есть pickle поэтому стало проще
df_start=pd.read_pickle(r'SQL  database\Service files\Генерационные слайды\df_start.pkl')

In [135]:
df_start.info()
# Как здорово что все на местах

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72139 entries, 0 to 72138
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         72139 non-null  int64         
 1   order_date       72139 non-null  datetime64[ns]
 2   customer_name    72139 non-null  object        
 3   customer_phone   72139 non-null  object        
 4   product_id       72139 non-null  int64         
 5   product_name     72139 non-null  object        
 6   quantity         72139 non-null  int64         
 7   total_amount     72139 non-null  float64       
 8   discount_amount  72139 non-null  float64       
 9   status           72139 non-null  object        
 10  comment          72139 non-null  object        
 11  delivery_date    72139 non-null  datetime64[ns]
 12  gift_card        72139 non-null  object        
 13  order_id_fk      72139 non-null  int64         
 14  customer_id_fk   72139 non-null  int64

In [160]:
# Оставляем только нужные колонки
df_start1=df_start[['order_id_fk', 'product_id','quantity', 'total_amount',
       'discount_amount']]

df_start2=df_start1.copy()

In [161]:
# Добавили процент скидки, будевые поля, и item_type. 
df_start2.loc[:,'discount_percentage']=round((df['discount_amount']/df['total_amount']*100),2)
df_start2.loc[:,['rental_id','subscribe_id']]=pd.NA
df_start2.loc[:,'item_type'] = 1 


In [ ]:
# Удаляю лишниe поля
#df_start2.drop(columns=['total_amount', 'discount_amount'], inplace=True)


#df_start2 = df_start2[['order_id_fk', 'product_id', 'quantity', 'discount_percentage',
#                      'item_type', 'rental_id', 'subscribe_id']]


#df_start2.columns = ['order_id', 'product_id', 'quantity', 'discount_percentage',
 #                   'item_type', 'rental_id', 'subscribe_id']






In [171]:
df_start2.to_pickle(r'SQL  database\Service files\Генерационные слайды\df_orderitem.pkl')

Создадим df для таблицы типов

In [173]:
df_item_type=pd.DataFrame({'description':['product','service','subscription','rental']})

In [ ]:
# И зальем
df_item_type.to_sql('itemtype',engine,schema='my_schema',if_exists='append',index=False)

In [ ]:
#пробуем залить orderitem
df_orderitem=pd.read_pickle(r'SQL  database\Service files\Генерационные слайды\df_orderitem.pkl')
#df_orderitem.to_sql('orderitem',engine,schema='my_schema',if_exists='append',index=False)

На этом этапе работы по заполнению баз считаю законченными. Далее будем переходить к витринам 
